<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343725359" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, MESSIDOR, CUSTOM CNN + EFFICIENTNETB0 =====
# Messidor is the last of three sources. Once all four architectures are done here,
# Stage 11 is fully complete (12/12 single-source models).

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
CUSTOM_LR = 1e-3
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT, MONITOR = 10, 1e-3, 1e-5, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD, MESSIDOR ONLY ----------
messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

# ---------- pairing verification gate, re-run here since this is a fresh rebuild ----------
is_im = ~messidor['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
im_check = messidor[is_im].copy()
im_check['im_num'] = im_check['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
im_check = im_check.sort_values('im_num').reset_index(drop=True)
im_check['pid'] = im_check.index // 2
sizes = im_check.groupby('pid').size()
full_pairs = im_check[im_check['pid'].isin(sizes[sizes == 2].index)]
agreement = full_pairs.groupby('pid')['grade'].apply(lambda g: g.iloc[0] == g.iloc[1])
print(f"Pairing agreement: {agreement.mean():.3f} (expect ~0.749)")
assert abs(agreement.mean() - 0.749) < 0.01, "Pairing evidence did not reproduce, stop and investigate"

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), "MESSIDOR IM PATIENT LEAKAGE"
    print(f"{tag} IM-style patient-leakage check: PASS ({len(im_df)} images, {im_df['patient_id'].nunique()} groups)")
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    print(f"{tag} date-style: {len(date_df)} images, image-level (no pairing signal, disclosed)")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

m_tr, m_va, m_te = split_messidor_mixed(messidor)
print(f"\nMessidor split: Train {len(m_tr)} | Val {len(m_va)} | Test {len(m_te)}")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=m_tr['grade'])
messidor_class_weight = {i: w for i, w in enumerate(cw)}
span = cw.max()/cw.min()
print(f"\nMessidor class weight span: {span:.1f}x (expect ~31.0x)")
print("Second-worst span after EyePACS. Custom CNN collapsed hard on EyePACS at this scale,")
print("expect a similar or worse outcome here given Messidor's grade 4 is even thinner (35 total).")

# ================================================================
# STAGE 11: MESSIDOR, CUSTOM CNN then EFFICIENTNETB0
# ================================================================

def make_source_gens(preprocess_fn, tr_df, va_df, te_df):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=5, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

stage11_results = []

# ---------- Custom CNN, Messidor ----------
tag = "ss_custom_messidor_dr"
tr, va, te = make_source_gens(None, m_tr, m_va, m_te)
model = build_custom_cnn()
model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
cbs = [EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{tag}.keras', monitor=MONITOR, save_best_only=True),
       CSVLogger(f'/kaggle/working/{tag}_log.csv', append=False)]
print(f"\n===== Custom CNN, Messidor (weight span {span:.1f}x) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=messidor_class_weight, callbacks=cbs, verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
if result[1] < 0.25:
    print("  ^ Near or below chance level (20% for 5 classes). Consistent with the EyePACS")
    print("    collapse pattern already observed for Custom CNN under severe class imbalance.")
stage11_results.append({'arch':'custom','source':'messidor','loss':result[0],'accuracy':result[1],'auc':result[2]})
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_messidor_custom.csv', index=False)
print("Checkpointed after Custom CNN.")
del model
tf.keras.backend.clear_session()

# ---------- EfficientNetB0, Messidor ----------
tag_p1 = "ss_eff_messidor_dr_phase1"
tag_p2 = "ss_eff_messidor_dr"
tr, va, te = make_source_gens(eff_pre, m_tr, m_va, m_te)
model, base = build_pretrained(EfficientNetB0)

base.trainable = False
model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== eff, messidor: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=messidor_class_weight,
          callbacks=[ModelCheckpoint(f'/kaggle/working/{tag_p1}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p1}_log.csv', append=False)], verbose=1)
print("eff, messidor Phase 1 saved.")

base.trainable = True
model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== eff, messidor: PHASE 2 (full fine-tune, up to 60 epochs) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=messidor_class_weight,
          callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint(f'/kaggle/working/{tag_p2}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p2}_log.csv', append=False)], verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag_p2} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
print("eff, messidor Phase 2 saved.")
stage11_results.append({'arch':'eff','source':'messidor','loss':result[0],'accuracy':result[1],'auc':result[2]})
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_messidor_custom_eff.csv', index=False)
print("Checkpointed after EfficientNetB0.")

print("\n===== MESSIDOR, Custom CNN + EfficientNetB0, COMPLETE =====")
print(pd.DataFrame(stage11_results).to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-20 14:41:54.929586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787236914.952216      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787236914.959970      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787236914.978351      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787236914.978370      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787236914.978373      22 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Pairing agreement: 0.749 (expect ~0.749)
WARNING [Messidor IM second]: stratified split failed, falling back to unstratified.
  sklearn error: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
Messidor IM-style patient-leakage check: PASS (687 images, 344 groups)
Messidor date-style: 1057 images, image-level (no pairing signal, disclosed)

Messidor split: Train 1219 | Val 262 | Test 263

Messidor class weight span: 31.0x (expect ~31.0x)
Second-worst span after EyePACS. Custom CNN collapsed hard on EyePACS at this scale,
expect a similar or worse outcome here given Messidor's grade 4 is even thinner (35 total).
Found 1219 validated image filenames belonging to 5 classes.
Found 262 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.


I0000 00:00:1787236929.776412      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787236929.782892      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



===== Custom CNN, Messidor (weight span 31.0x) =====
Epoch 1/60


...

39/39 [==============================] - 52s 1s/step - loss: 1.7145 - accuracy: 0.2986 - auc: 0.6016 - val_loss: 1.4093 - val_accuracy: 0.2137 - val_auc: 0.7224
Epoch 2/60
39/39 [==============================] - 31s 782ms/step - loss: 1.6429 - accuracy: 0.3322 - auc: 0.6307 - val_loss: 1.3146 - val_accuracy: 0.5802 - val_auc: 0.8079
Epoch 3/60
39/39 [==============================] - 30s 780ms/step - loss: 1.5557 - accuracy: 0.3618 - auc: 0.6530 - val_loss: 1.2188 - val_accuracy: 0.5802 - val_auc: 0.8002
Epoch 4/60
39/39 [==============================] - 31s 803ms/step - loss: 1.5748 - accuracy: 0.3568 - auc: 0.6543 - val_loss: 1.4328 - val_accuracy: 0.5763 - val_auc: 0.7557
Epoch 5/60
39/39 [==============================] - 31s 784ms/step - loss: 1.5903 - accuracy: 0.3667 - auc: 0.6481 - val_loss: 1.6996 - val_accuracy: 0.0344 - val_auc: 0.6011
Epoch 6/60
39/39 [==============================] - 31s 786ms/step - loss: 1.5756 - accuracy: 0.3167 - auc: 0.6489 - val_loss: 1.7224 - val

E0000 00:00:1787237242.058446      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


39/39 [==============================] - 32s 669ms/step - loss: 1.6371 - accuracy: 0.2888 - auc: 0.6468 - val_loss: 1.3322 - val_accuracy: 0.4084 - val_auc: 0.7727
Epoch 2/10
39/39 [==============================] - 23s 596ms/step - loss: 1.3865 - accuracy: 0.4011 - auc: 0.7479 - val_loss: 1.4278 - val_accuracy: 0.3702 - val_auc: 0.7138
Epoch 3/10
39/39 [==============================] - 23s 595ms/step - loss: 1.2658 - accuracy: 0.4610 - auc: 0.7933 - val_loss: 1.3466 - val_accuracy: 0.3855 - val_auc: 0.7495
Epoch 4/10
39/39 [==============================] - 24s 613ms/step - loss: 1.1999 - accuracy: 0.4487 - auc: 0.7894 - val_loss: 1.2871 - val_accuracy: 0.4618 - val_auc: 0.7701
Epoch 5/10
39/39 [==============================] - 23s 595ms/step - loss: 1.1600 - accuracy: 0.4996 - auc: 0.8156 - val_loss: 1.1724 - val_accuracy: 0.3435 - val_auc: 0.7956
Epoch 6/10
39/39 [==============================] - 23s 597ms/step - loss: 1.2163 - accuracy: 0.4668 - auc: 0.8076 - val_loss: 1.2955 - 

E0000 00:00:1787237496.931743      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


39/39 [==============================] - 77s 864ms/step - loss: 1.7595 - accuracy: 0.2469 - auc: 0.5591 - val_loss: 0.9586 - val_accuracy: 0.5802 - val_auc: 0.8806
Epoch 2/60
39/39 [==============================] - 30s 763ms/step - loss: 1.5713 - accuracy: 0.2896 - auc: 0.6065 - val_loss: 0.9682 - val_accuracy: 0.5649 - val_auc: 0.8714
Epoch 3/60
39/39 [==============================] - 30s 769ms/step - loss: 1.5383 - accuracy: 0.3084 - auc: 0.6288 - val_loss: 0.9872 - val_accuracy: 0.5534 - val_auc: 0.8627
Epoch 4/60
39/39 [==============================] - 30s 779ms/step - loss: 1.4067 - accuracy: 0.3454 - auc: 0.6646 - val_loss: 1.0324 - val_accuracy: 0.5115 - val_auc: 0.8466
Epoch 5/60
39/39 [==============================] - 31s 782ms/step - loss: 1.4046 - accuracy: 0.3511 - auc: 0.6808 - val_loss: 1.0605 - val_accuracy: 0.5038 - val_auc: 0.8371
Epoch 6/60
39/39 [==============================] - 31s 788ms/step - loss: 1.3509 - accuracy: 0.3782 - auc: 0.7057 - val_loss: 1.0758 - 